# 06 — Property Value Distribution Buckets

**Phase:** Processed Output Aggregation  
**Notebook goal:** Create a compact binned summary of assessed property value percentage changes from the local row-level distribution file generated in Notebook 05.

---

## Purpose

Notebook 05 processed the full City of Vancouver Property Tax Report and wrote two files to `data/processed/`:

- `property_value_change_summary.csv` — a quality-control summary with one row per derived metric. This file is small and tracked in Git.
- `property_value_change_distribution.csv` — a row-level file containing the four derived property value change columns for all 1,552,663 property records. This file is approximately 68.47 MB and is **excluded from Git** due to its size.

This notebook reads only the `percentage_value_change` column from the local distribution file and assigns each record to a percentage change bucket. The result is a small aggregated CSV — `property_value_change_distribution_bins.csv` — that is suitable for version control and downstream visualisation.

> **Important caveat:** BC Assessment assessed values are not MLS sale prices, transaction prices, or market values. They are administrative valuations used as the basis for property tax calculations in British Columbia. Year-over-year changes in assessed value do not directly measure market appreciation or depreciation.

> **Scope:** This notebook does not modify, recreate, or delete the large distribution file. It reads from it once and writes only the small binned summary.

## 2. Imports and Paths

We use `pathlib` for cross-platform path handling and `pandas` for all tabular operations. All paths are derived from `PROJECT_ROOT` so the notebook works regardless of where it is launched from.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT       = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
DISTRIBUTION_PATH  = PROCESSED_DATA_DIR / 'property_value_change_distribution.csv'
BINS_OUTPUT_PATH   = PROCESSED_DATA_DIR / 'property_value_change_distribution_bins.csv'

print(f'PROJECT_ROOT       : {PROJECT_ROOT}')
print(f'PROCESSED_DATA_DIR : {PROCESSED_DATA_DIR}')
print(f'DISTRIBUTION_PATH  : {DISTRIBUTION_PATH}')
print(f'BINS_OUTPUT_PATH   : {BINS_OUTPUT_PATH}')

## 3. Input Validation

Before loading data, we confirm the distribution file exists and report its size. Only the `percentage_value_change` column is loaded — the other three derived columns are not needed for bucketing and loading only the required column keeps memory usage low.

If the assert below fails, run Notebook 05 with `RUN_FULL_PROCESSING = True` to generate the distribution file locally.

In [ ]:
assert DISTRIBUTION_PATH.exists(), (
    f'Distribution file not found: {DISTRIBUTION_PATH}\n'
    'Ensure Notebook 05 has been run with RUN_FULL_PROCESSING = True '
    'to generate property_value_change_distribution.csv locally.'
)

file_size_mb = DISTRIBUTION_PATH.stat().st_size / (1024 ** 2)
print(f'File found    : {DISTRIBUTION_PATH.name}')
print(f'File size     : {file_size_mb:.2f} MB')

COL_PCT_CHANGE = 'percentage_value_change'

df = pd.read_csv(DISTRIBUTION_PATH, usecols=[COL_PCT_CHANGE])

print(f'Rows loaded   : {len(df):,}')
print(f'Column loaded : {list(df.columns)}')
print(f'Dtype         : {df[COL_PCT_CHANGE].dtype}')
print(f'Non-null      : {df[COL_PCT_CHANGE].notna().sum():,}')
print(f'Missing       : {df[COL_PCT_CHANGE].isna().sum():,}')

## 4. Bucket Definitions

Ten buckets cover the full range of `percentage_value_change` values, including a catch-all for records where the column could not be computed.

Bin edges use `right=True` (the `pd.cut` default), so each interval is left-open and right-closed: `(a, b]`. The `include_lowest=True` flag closes the leftmost edge so that any value at exactly `−90%` (the theoretical minimum for a total loss) is captured. The `Missing / not calculable` label covers records where the previous-year total was null, zero, or negative and a percentage change could not be defined.

In [ ]:
BUCKET_MISSING = 'Missing / not calculable'

BUCKET_EDGES = [float('-inf'), -50.0, -20.0, -10.0, 0.0, 5.0, 10.0, 20.0, 50.0, float('inf')]
BUCKET_LABELS = [
    'Below -50%',
    '-50% to -20%',
    '-20% to -10%',
    '-10% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 20%',
    '20% to 50%',
    'Above 50%',
]

BUCKET_ORDER = BUCKET_LABELS + [BUCKET_MISSING]

print(f'Numeric buckets : {len(BUCKET_LABELS)}')
print(f'Total buckets   : {len(BUCKET_ORDER)} (including missing)')
for i, b in enumerate(BUCKET_ORDER, 1):
    print(f'  {i:>2}. {b}')

## 5. Binned Summary

`pd.cut` assigns each non-null value to a bucket. The resulting Categorical is converted to `object` dtype so that `NaN` entries can be filled with the `Missing / not calculable` label using `.fillna()`. The final summary DataFrame is reindexed against `BUCKET_ORDER` to guarantee the logical ordering defined in Section 4, not alphabetical ordering.

Two assertions confirm correctness before the output is exported:

- **Count check:** the sum of `property_count` must equal the number of rows loaded.
- **Percentage check:** `percentage_of_records` must sum to approximately 100% (within 0.01 percentage points to account for floating-point rounding).

In [ ]:
buckets = pd.cut(
    df[COL_PCT_CHANGE],
    bins=BUCKET_EDGES,
    labels=BUCKET_LABELS,
    right=True,
    include_lowest=True,
)

bucket_str = buckets.astype(object).fillna(BUCKET_MISSING)

counts  = bucket_str.value_counts()
summary = counts.reindex(BUCKET_ORDER, fill_value=0).reset_index()
summary.columns = ['bucket', 'property_count']
summary['percentage_of_records'] = summary['property_count'] / len(df) * 100

total_count = summary['property_count'].sum()
assert total_count == len(df), (
    f'Count mismatch: sum={total_count:,}, expected={len(df):,}'
)
pct_sum = summary['percentage_of_records'].sum()
assert abs(pct_sum - 100.0) < 0.01, (
    f'Percentages do not sum to approximately 100%: {pct_sum:.4f}%'
)

print('Validation passed.')
print(f'Total rows accounted for : {total_count:,}')
print(f'Percentage sum           : {pct_sum:.4f}%')
print()
display(summary)

## 6. Export

The binned summary is written to `data/processed/property_value_change_distribution_bins.csv`. This file is small (10 rows, 3 columns) and is suitable for version control.

In [ ]:
summary.to_csv(BINS_OUTPUT_PATH, index=False)

print(f'Written : {BINS_OUTPUT_PATH.name}')
print(f'Path    : {BINS_OUTPUT_PATH}')
print(f'Rows    : {len(summary):,}')

## 7. Output Validation

Six checks confirm that the output file is correct and complete before this notebook is considered done. These checks are designed to be re-runnable: if the notebook is re-executed, they confirm that the re-generated output is still valid.

In [ ]:
assert BINS_OUTPUT_PATH.exists(), f'Output file not found: {BINS_OUTPUT_PATH}'
print('Output file exists              : OK')

reload = pd.read_csv(BINS_OUTPUT_PATH)

assert len(reload) > 0, 'Output file is empty'
print(f'Output file is non-empty        : {len(reload):,} rows — OK')

EXPECTED_COLS = {'bucket', 'property_count', 'percentage_of_records'}
actual_cols   = set(reload.columns.tolist())
assert EXPECTED_COLS == actual_cols, (
    f'Column mismatch.\n  Expected : {sorted(EXPECTED_COLS)}\n  Got      : {sorted(actual_cols)}'
)
print('Expected columns present        : OK')

count_sum = reload['property_count'].sum()
assert count_sum == len(df), (
    f'Count sum mismatch: {count_sum:,} != {len(df):,}'
)
print(f'Property counts sum correctly   : {count_sum:,} — OK')

assert reload['bucket'].nunique() == len(reload), 'Duplicate bucket names found'
print('No duplicate bucket names       : OK')

expected_buckets = set(BUCKET_ORDER)
actual_buckets   = set(reload['bucket'].tolist())
assert expected_buckets == actual_buckets, (
    f'Bucket name mismatch.\n'
    f'  Missing from output : {expected_buckets - actual_buckets}\n'
    f'  Unexpected in output: {actual_buckets - expected_buckets}'
)
print('All expected buckets present    : OK')

print()
print('All output validation checks passed.')

## 8. Final Status

This cell prints a clear confirmation of what this notebook does, what it produced, and what it does not do.

In [ ]:
print('=' * 70)
print('NOTEBOOK STATUS')
print('=' * 70)
print()
print('Input')
print(f'  File : {DISTRIBUTION_PATH.name}')
print(f'  Note : local file, excluded from Git due to size (~68 MB)')
print(f'  Rows : {len(df):,}')
print()
print('Output')
print(f'  File : {BINS_OUTPUT_PATH.name}')
print(f'  Note : small aggregated file, suitable for version control')
print(f'  Rows : {len(summary):,} (one row per bucket)')
print()
print('This notebook does not make causal claims about housing supply,')
print('permitting activity, or market prices. Assessed values are')
print('administrative property valuations, not MLS sale prices.')
print('=' * 70)